# 02 — Data Cleaning Pipeline
## Overview
This notebook implements the cleaning and standardization pipeline
for the higher education outcomes dataset.

Main objectives:

- standardize schemas
- normalize categorical values
- resolve structural inconsistencies
- validate metric integrity
- generate analysis-ready tables

The notebook assumes that the raw data audit performed in
`01_data_audit.ipynb` has already been completed.

In [1]:
from notebook_utils import ensure_repo_root

# Establish the repository root as the working directory for this notebook
ensure_repo_root()

WindowsPath('C:/Github/higher-education-outcomes-analysis')

In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.utils.data_utils import load_data

enrollment = load_data("enrollment")
programs = load_data("programs")
offering = load_data("offering")


In [3]:
from src.config.mappings import (
    CANONICAL_MAPPINGS,
    SHIFT_TYPO_MAP,
    DELIVERY_MODE_TYPO_MAP,
    WEEKDAY_TYPO_MAP,
)

from src.cleaning import (
    drop_columns,
    normalize_text_columns,
    correct_typo_variants,
    apply_canonical_taxonomy,
    reconcile_metric_totals
)

drop columns

In [4]:
metadata_cols = [
    "shift",
    "weekday",
    "schedule_time",
    "delivery_mode",
    "campus",
]

In [5]:
enrollment_cleaned = drop_columns(
                     df=enrollment,
                     cols_to_drop=metadata_cols
)

Columns succesfully dropped: ['shift', 'weekday', 'schedule_time', 'delivery_mode', 'campus']


normalize text columns

In [6]:
text_cols = [
    "shift",
    "weekday",
    "schedule_time",
    "delivery_mode",
]

In [7]:
offering_cleaned = normalize_text_columns(
                   df=offering,
                   text_cols=text_cols
)

before/after

In [8]:
display(offering[text_cols].nunique())
display(offering_cleaned[text_cols].nunique())

shift             6
weekday          15
schedule_time    38
delivery_mode    26
dtype: int64

shift             4
weekday           8
schedule_time    34
delivery_mode    17
dtype: int64

correct typographical problems

delivery_mode
mapping into "presencial" and "virtual"

In [9]:
# Still not modifying the offering_cleaned dataframe
delivery_mode_clean = correct_typo_variants(offering_cleaned, 
                      column="delivery_mode", 
                      typo_map=DELIVERY_MODE_TYPO_MAP, 
                      unmapped="nan",
                      verbose=True,
                      normalize_func=None,
                      )["delivery_mode"] # taking only the cleaned column

Column: 'delivery_mode'
Number of unique categories before typo correction: 17
Number of unique categories after typo correction: 3
['presencial' nan 'virtual']


remaining fallback:

In [10]:
offering_cleaned[delivery_mode_clean.isna()]["delivery_mode"].value_counts()

delivery_mode
4 hs presencial y 2 virtual              26
virtual (con encuentros presenciales)     5
4 presencial y 2 virtual                  5
presencial y 2 hs virtual                 4
4 hs presenciales y 2 virtuales           4
4 hs presencial 2 virtual                 3
4 presencial, 2 virtual                   3
2 presencial y 4 virtual                  2
4 presenciales y 2 virtuales              1
3 hs presencial 3 virtual                 1
2 hs presencial y 2 virtual               1
2 hs practicas, 4 presenciales            1
3 presencial y 3 virtual                  1
Name: count, dtype: int64

because every value is suitable to be assigned to a hybrid delivery mode:

In [11]:
offering_cleaned["delivery_mode"] = delivery_mode_clean.fillna("hibrida")
offering_cleaned["delivery_mode"].unique()

array(['presencial', 'hibrida', 'virtual'], dtype=object)

shift

In [12]:
offering_cleaned = correct_typo_variants(
                   df=offering_cleaned,
                   column="shift",
                   typo_map=SHIFT_TYPO_MAP,
                   unmapped="ignore",
                   verbose=True,
                   normalize_func=None,
)

Column: 'shift'
Number of unique categories before typo correction: 4
Number of unique categories after typo correction: 3
['noche' 'manana' 'tarde']


weekday

In [13]:
offering_cleaned = correct_typo_variants(
                   df=offering_cleaned,
                   column="weekday",
                   typo_map=WEEKDAY_TYPO_MAP,
                   unmapped="ignore",
                   verbose=True,
                   normalize_func=None,
)

Column: 'weekday'
Number of unique categories before typo correction: 8
Number of unique categories after typo correction: 6
['martes' 'sabado' 'jueves' 'lunes' 'viernes' 'miercoles']


canonical taxonomy

In [14]:
for column, mapping in CANONICAL_MAPPINGS.items():
    offering_cleaned = apply_canonical_taxonomy(
                       df=offering_cleaned,
                       column=column,
                       canonical_map=mapping,
    )

Column: 'shift'
Unique categories after applying canonical taxonomy: ['night' 'morning' 'afternoon']
Column: 'delivery_mode'
Unique categories after applying canonical taxonomy: ['on_site' 'hybrid' 'online']
Column: 'weekday'
Unique categories after applying canonical taxonomy: ['tuesday' 'saturday' 'thursday' 'monday' 'friday' 'wednesday']


workload to int

In [15]:
offering_cleaned["workload"] = offering_cleaned["workload"].astype("Int64")

metric reconciliation

In [16]:
metric_cols = [
    "dropout_count",
    "insufficient_count",
    "free_status_count",
    "promoted_completion_count",
    "regular_completion_count",
]

In [17]:
enrollment_cleaned = reconcile_metric_totals(
    df=enrollment_cleaned,
    component_columns=metric_cols,
    reported_column="total_enrollment",
    tolerance=2,
    overwrite=True,
)

keeps the reported metric, while replaces it in total_enrollment with the computed one, always that the difference<=2

In [18]:
enrollment_cleaned.loc[
    enrollment_cleaned["total_enrollment_difference"] != 0,
    [
        "reported_total_enrollment",
        "computed_total_enrollment",
        "reconciled_total_enrollment",
        "total_enrollment_difference",
    ],
]

,reported_total_enrollment,computed_total_enrollment,reconciled_total_enrollment,total_enrollment_difference
35,19,18,18,1
101,51,49,49,2
111,58,57,57,1
145,57,56,56,1


join tables

In [19]:
data = enrollment_cleaned.merge(offering_cleaned, on=["course_code", "section"], how="left")
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 303 entries, 0 to 302
Data columns (total 20 columns):
 #   Column                       Non-Null Count  Dtype 
---  ------                       --------------  ----- 
 0   course_name                  303 non-null    object
 1   section                      303 non-null    int64 
 2   total_enrollment             303 non-null    int64 
 3   dropout_count                303 non-null    int64 
 4   insufficient_count           303 non-null    int64 
 5   free_status_count            303 non-null    int64 
 6   promoted_completion_count    303 non-null    int64 
 7   regular_completion_count     303 non-null    int64 
 8   course_code                  303 non-null    object
 9   program_code                 303 non-null    object
 10  reported_total_enrollment    303 non-null    int64 
 11  computed_total_enrollment    303 non-null    int64 
 12  total_enrollment_difference  303 non-null    int64 
 13  reconciled_total_enrollment  303 no

A small subset of unmatched course-section observations was excluded during the cleaning stage due to unresolved operational metadata gaps after integration with the canonical Offering source.

In [20]:
data = data.dropna()
data.info()

<class 'pandas.core.frame.DataFrame'>
Index: 297 entries, 0 to 302
Data columns (total 20 columns):
 #   Column                       Non-Null Count  Dtype 
---  ------                       --------------  ----- 
 0   course_name                  297 non-null    object
 1   section                      297 non-null    int64 
 2   total_enrollment             297 non-null    int64 
 3   dropout_count                297 non-null    int64 
 4   insufficient_count           297 non-null    int64 
 5   free_status_count            297 non-null    int64 
 6   promoted_completion_count    297 non-null    int64 
 7   regular_completion_count     297 non-null    int64 
 8   course_code                  297 non-null    object
 9   program_code                 297 non-null    object
 10  reported_total_enrollment    297 non-null    int64 
 11  computed_total_enrollment    297 non-null    int64 
 12  total_enrollment_difference  297 non-null    int64 
 13  reconciled_total_enrollment  297 non-nul

In [21]:
data = data.merge(programs, on="program_code", how="left")
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 297 entries, 0 to 296
Data columns (total 21 columns):
 #   Column                       Non-Null Count  Dtype 
---  ------                       --------------  ----- 
 0   course_name                  297 non-null    object
 1   section                      297 non-null    int64 
 2   total_enrollment             297 non-null    int64 
 3   dropout_count                297 non-null    int64 
 4   insufficient_count           297 non-null    int64 
 5   free_status_count            297 non-null    int64 
 6   promoted_completion_count    297 non-null    int64 
 7   regular_completion_count     297 non-null    int64 
 8   course_code                  297 non-null    object
 9   program_code                 297 non-null    object
 10  reported_total_enrollment    297 non-null    int64 
 11  computed_total_enrollment    297 non-null    int64 
 12  total_enrollment_difference  297 non-null    int64 
 13  reconciled_total_enrollment  297 no